In [5]:
from strange_loop.replay import Loop, stage


@stage
def double(number):
    return number * 2


@stage
def increment(number):
    return number + 1


async def calculate(ctx, number):
    incr = await increment(number=number)
    dbl = await double(number=incr)
    return dbl


loop = Loop()
answer = await loop.run(calculate, number=10)
print(answer.value)  # 22


22


In [6]:
print(loop.trace())


01  calculate  start
02  calculate  start increment()
03  calculate  finish increment() -> calculate/increment/1@1
04  calculate  start double()
05  calculate  finish double() -> calculate/double/1@1
06  calculate  return calculate/double/1@1


In [7]:
parent = loop.input(answer, "number")
root = loop.input(parent, "number")

print(answer.actor, answer.value)  # double 22
print(parent.actor, parent.value)  # increment 11
print(root.actor, root.value)  # input 10


double 22
increment 11
<input> 10


In [ ]:
from strange_loop.replay import Batch, Context, Loop, Version, batch_stage


@batch_stage
def uppercase(batch: Batch) -> dict[str, str]:
    return {item.key: item.inputs["text"].upper() for item in batch}


async def shout(ctx: Context, source: Version) -> Version:
    return await uppercase(text=source)


loop = Loop()
results = await loop.map(shout, source={"a": "one", "b": "two", "c": "three"})
print([r.value for r in results.values()])  # ['ONE', 'TWO', 'THREE']
print(loop.batches)


  + Exception Group Traceback (most recent call last):
  |   File "/Users/thomas/code/relaxed-constraints/strange-loop/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3821, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/var/folders/cp/r86_5xy15c564s3wsczlqgrc0000gn/T/ipykernel_92875/977629943.py", line 14, in <module>
  |     results = await loop.map(shout, source="test")
  |               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/Users/thomas/code/relaxed-constraints/strange-loop/src/strange_loop/replay.py", line 333, in map
  |     missing = self._pending.pop(queue)
  |                ^^^^^^^^^^^^^^^^^^^
  |   File "/Users/thomas/.local/share/uv/python/cpython-3.12.9-macos-aarch64-none/lib/python3.12/asyncio/taskgroups.py", line 71, in __aexit__
  |     return await self._aexit(et, exc)
  |            ^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/Users/thomas/.local/share/uv/python/cpython-3.12.9-macos-aarch64-non